### EDA of Q1 2025

**Objective**: Find the problem statement & the research questions for the project proposal.

#### 1 - Scope and Configuration

The next cell defines a single frozen configuration object (cfg) that holds every project parameter in one place: the data paths and period, the exposure filters, the prediction horizons H and minimum warning times L, the critical SMART attributes, and the cost ratios.

It also sets the random seeds for reproducibility, creates the cache and figure folders, and checks that the daily CSV files exist.


Later parts of the notebook read from cfg, so moving from Q1 to the full year only requires editing this cell.

In [6]:
from pathlib import Path
from dataclasses import dataclass, field
import random
import numpy as np

@dataclass(frozen=True)
class Config:
    # --- Data location and period ---
    data_dir: Path = Path("../data/data_Q1_2025")      # folder with the daily CSVs (YYYY-MM-DD.csv)
    start_date: str = "2025-01-01"
    end_date: str = "2025-03-31"                    # for the full year: "2025-12-31"
    cache_dir: Path = Path("cache")                 # intermediate results
    fig_dir: Path = Path("figures")

    # --- Scope ---
    hdd_only: bool = True                           # SSD filter
    min_drive_days_per_model: int = 10_000          # exposure filter for AFR reliability (Annualized Failure Rate)
    min_failures_per_model: int = 5                 # minimum failures to report a model-level rate

    # --- Prediction design ---
    horizons_H: tuple = (7, 14, 30)                 # "fails within H days"
    min_warning_L: tuple = (3, 7, 14)               # minimum actionable lead time (days)
    prefailure_window: int = 60                     # days before failure 

    # --- SMART attributes to focus on ---
    critical_smart: tuple = (5, 187, 188, 197, 198)
    other_smart: tuple = (9, 194)                   # power-on hours, temperature

    # --- Cost scenarios (ratio R/M, replacement vs failure cost) ---
    cost_ratios: tuple = (0.05, 0.2, 0.5)

    # --- Reproducibility ---
    seed: int = 42

    # --- Derived helpers ---
    @property
    def critical_cols_raw(self):
        return [f"smart_{i}_raw" for i in self.critical_smart]

    @property
    def critical_cols_norm(self):
        return [f"smart_{i}_normalized" for i in self.critical_smart]

    @property
    def other_cols_raw(self):
        return [f"smart_{i}_raw" for i in self.other_smart]

cfg = Config()

# --- Setup ---
random.seed(cfg.seed)
np.random.seed(cfg.seed)
cfg.cache_dir.mkdir(parents=True, exist_ok=True)
cfg.fig_dir.mkdir(parents=True, exist_ok=True)

# --- Sanity check: are the input files where we expect them? ---
files = sorted(cfg.data_dir.glob("*.csv"))
assert files, f"No CSV files found in {cfg.data_dir.resolve()}"

print(f"Period          : {cfg.start_date} -> {cfg.end_date}")
print(f"Daily files     : {len(files)}  ({files[0].name} ... {files[-1].name})")
print(f"Horizons H      : {cfg.horizons_H}")
print(f"Min warning L   : {cfg.min_warning_L}")
print(f"Critical SMART  : {cfg.critical_smart}")
print(f"HDD only        : {cfg.hdd_only}")

Period          : 2025-01-01 -> 2025-03-31
Daily files     : 90  (2025-01-01.csv ... 2025-03-31.csv)
Horizons H      : (7, 14, 30)
Min warning L   : (3, 7, 14)
Critical SMART  : (5, 187, 188, 197, 198)
HDD only        : True


#### 2 - Loading and Schema audit

This cell first reads only the header of each daily CSV to check whether the schema is consistent across files, and lists any columns that are missing in some of them. It then converts all the daily files into a single compressed Parquet file, restricted to the configured date range, so later parts of the notebook load quickly. 

Next, it reports the dataset's size and classifies every column as identifier, target, static, or SMART (raw and normalized), and checks that the SMART columns set in cfg actually exist. It also compares the number of distinct dates with the expected number of days and the number of files, which is an early check for missing days. 

Finally, it shows the most common drive models and a few sample rows, which helps us to design the HDD/SSD filter and confirm the data types.

In [9]:
import re
import duckdb
import pandas as pd
from collections import Counter

con = duckdb.connect()
parquet_path = cfg.cache_dir / "drivestats_q1_2025.parquet"

# --- 1. Schema consistency across daily files (header only, fast) ---
headers = {f.name: tuple(pd.read_csv(f, nrows=0).columns) for f in files}
schema_variants = Counter(headers.values())
all_cols = list(dict.fromkeys(c for h in headers.values() for c in h)) 

print(f"Files: {len(files)} | distinct schemas: {len(schema_variants)} | union columns: {len(all_cols)}")
if len(schema_variants) > 1:
    common = set.intersection(*[set(h) for h in headers.values()])
    print("Columns missing in some files:", sorted(set(all_cols) - common))

# --- 2. Convert CSVs -> one Parquet file (skipped if already cached) ---
if not parquet_path.exists():
    types = {c: "DOUBLE" for c in all_cols if c.startswith("smart_")}
    types.update({"date": "DATE", "serial_number": "VARCHAR", "model": "VARCHAR",
                  "capacity_bytes": "BIGINT", "failure": "TINYINT"})
    file_list = "[" + ",".join(f"'{f.as_posix()}'" for f in files) + "]"
    con.execute(f"""
        COPY (
            SELECT * FROM read_csv({file_list}, union_by_name=true, types={types})
            WHERE date BETWEEN DATE '{cfg.start_date}' AND DATE '{cfg.end_date}'
        ) TO '{parquet_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
con.execute(f"CREATE OR REPLACE VIEW raw AS SELECT * FROM read_parquet('{parquet_path.as_posix()}')")

# --- 3. Basic shape ---
n_rows = con.execute("SELECT COUNT(*) FROM raw").fetchone()[0]
schema = con.execute("DESCRIBE raw").df()[["column_name", "column_type"]]
print(f"\nRows: {n_rows:,} | Columns: {len(schema)} | Parquet size: {parquet_path.stat().st_size/1e9:.2f} GB")

# --- 4. Column classification ---
cols = schema["column_name"].tolist()
target_col  = ["failure"]
id_cols     = [c for c in ["date", "serial_number", "model"] if c in cols]
smart_raw   = [c for c in cols if re.fullmatch(r"smart_\d+_raw", c)]
smart_norm  = [c for c in cols if re.fullmatch(r"smart_\d+_normalized", c)]
static_cols = [c for c in cols if c not in id_cols + target_col + smart_raw + smart_norm]

print(f"\nIdentifiers : {id_cols}")
print(f"Target      : {target_col}")
print(f"Static/other: {static_cols}")
print(f"SMART raw   : {len(smart_raw)} | SMART normalized: {len(smart_norm)}")

# Are the config's critical SMART columns present?
missing = [c for c in cfg.critical_cols_raw + cfg.critical_cols_norm + cfg.other_cols_raw if c not in cols]
print("Missing configured SMART columns:", missing or "none")

# --- 5. Date coverage vs number of files ---
d = con.execute("SELECT MIN(date), MAX(date), COUNT(DISTINCT date) FROM raw").fetchone()
expected_days = (pd.to_datetime(cfg.end_date) - pd.to_datetime(cfg.start_date)).days + 1
print(f"\nDates: {d[0]} -> {d[1]} | distinct days: {d[2]} | expected: {expected_days} | files: {len(files)}")

# --- 6. Quick look: model names (to design the HDD/SSD filter and manufacturer parsing) ---
models = con.execute("""
    SELECT model, ROUND(MAX(capacity_bytes)/1e12, 1) AS capacity_tb,
           COUNT(DISTINCT serial_number) AS drives
    FROM raw GROUP BY model ORDER BY drives DESC
""").df()
print(f"\nDistinct models: {len(models)}")
display(models.head(20))

# --- 7. Sample rows and dtypes ---
display(con.execute(f"SELECT {', '.join(id_cols + target_col + static_cols + cfg.critical_cols_raw)} "
                    "FROM raw USING SAMPLE 5").df())
display(schema.assign(kind=lambda x: x.column_name.map(
    lambda c: "target" if c in target_col else "id" if c in id_cols
    else "smart_raw" if c in smart_raw else "smart_norm" if c in smart_norm else "static")
).groupby(["kind", "column_type"]).size().rename("n_cols").reset_index())

Files: 90 | distinct schemas: 1 | union columns: 197

Rows: 27,799,986 | Columns: 197 | Parquet size: 0.81 GB

Identifiers : ['date', 'serial_number', 'model']
Target      : ['failure']
Static/other: ['capacity_bytes', 'datacenter', 'cluster_id', 'vault_id', 'pod_id', 'pod_slot_num', 'is_legacy_format']
SMART raw   : 93 | SMART normalized: 93
Missing configured SMART columns: none

Dates: 2025-01-01 -> 2025-03-31 | distinct days: 90 | expected: 90 | files: 90

Distinct models: 76


,model,capacity_tb,drives
0,TOSHIBA MG08ACA16TA,16.0,40273
1,TOSHIBA MG07ACA14TA,14.0,37705
2,WDC WUH722222ALE6L4,22.0,34857
3,ST16000NM001G,16.0,33817
4,WDC WUH721816ALE6L4,16.0,26428
5,ST12000NM0008,12.0,19145
6,ST8000NM0055,8.0,13533
7,HGST HUH721212ALE604,12.0,13435
8,ST12000NM001G,12.0,13214
9,TOSHIBA MG10ACA20TE,20.0,10814


,date,serial_number,model,failure,capacity_bytes,datacenter,cluster_id,vault_id,pod_id,pod_slot_num,is_legacy_format,smart_5_raw,smart_187_raw,smart_188_raw,smart_197_raw,smart_198_raw
0,2025-01-02,ZA1812B1,ST8000NM0055,0,8001563222016,sac0,000,1091,06,0000,False,0.0,0.0,0.0,0.0,0.0
1,2025-01-03,X0GE8XAC,WDC WUH721414ALE6L4,0,14000519643136,sac2,000,1149,04,0012,False,0.0,NaN,NaN,0.0,0.0
2,2025-01-03,ZHZ3HZ33,ST14000NM0138,0,14000519643136,sac2,000,1156,15,0006,False,0.0,0.0,0.0,0.0,0.0
3,2025-01-03,X0GEU6LC,WDC WUH721414ALE6L4,0,14000519643136,sac0,000,1158,14,0016,False,0.0,NaN,NaN,0.0,0.0
4,2025-01-03,11J0A0MCF97G,TOSHIBA MG07ACA14TA,0,14000519643136,sac2,000,1160,08,0051,False,0.0,NaN,NaN,0.0,0.0


,kind,column_type,n_cols
0,id,DATE,1
1,id,VARCHAR,2
2,smart_norm,DOUBLE,93
3,smart_raw,DOUBLE,93
4,static,BIGINT,2
5,static,BOOLEAN,1
6,static,VARCHAR,4
7,target,TINYINT,1
